# House Prices Prediction

This project aims to predict house prices using machine learning.

# Setup and Data Loading

In [ ]:
import numpy as np 
import pandas as pd 
import os

In [ ]:
train_raw = pd.read_csv('data/train.csv')
test_raw = pd.read_csv('data/test.csv')

train = train_raw.copy()
test = test_raw.copy()
train.shape
train.info()
train.describe()
#check N/A
missing = train.isnull().sum().sort_values(ascending=False)
missing.head(20)

# Data Preprocessing

In [ ]:
#Handling missing values
nones = ['FireplaceQu',
    'GarageQual', 'GarageCond', 'GarageType', 'GarageFinish',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']
for col in nones:
    train[col] = train[col].fillna('None')
    test[col] = test[col].fillna('None')

medians = ['LotFrontage', 'GarageYrBlt', 'MasVnrArea']
for col in medians:
    train[col] = train[col].fillna(train[col].median())
    test[col] = test[col].fillna(test[col].median())

modes = ['MasVnrType', 'Electrical']
for col in modes:
    train[col] = train[col].fillna(train[col].mode()[0])
    test[col] = test[col].fillna(test[col].mode()[0])

In [ ]:
#Feature construction
#1. Total floor area
train['TotalSF'] = train['TotalBsmtSF'] + train['1stFlrSF'] + train['2ndFlrSF']
test['TotalSF'] = test['TotalBsmtSF'] + test['1stFlrSF'] + test['2ndFlrSF']

#2. House age
train['HouseAge'] = train['YrSold'] - train['YearBuilt']
test['HouseAge'] = test['YrSold'] - test['YearBuilt']

#3. Reconstruction age
train['RemodAge'] = train['YrSold'] - train['YearRemodAdd']
test['RemodAge'] = test['YrSold'] - test['YearRemodAdd']

#4. Whether reconscturction
train['HasRemod'] = (train['YearRemodAdd'] != train['YearBuilt']).astype(int)
test['HasRemod'] = (test['YearRemodAdd'] != test['YearBuilt']).astype(int)

#5. Facilities
train['HasGarage'] = (train['GarageArea'] > 0).astype(int)
test['HasGarage'] = (test['GarageArea'] > 0).astype(int)

train['HasBsmt'] = (train['TotalBsmtSF'] > 0).astype(int)
test['HasBsmt'] = (test['TotalBsmtSF'] > 0).astype(int)

train['HasFireplace'] = (train['Fireplaces'] > 0).astype(int)
test['HasFireplace'] = (test['Fireplaces'] > 0).astype(int)

# Baseline model Setiing

In [ ]:
#Define model
y = np.log1p(train['SalePrice'])

train_features = train.drop('SalePrice', axis=1)
test_features = test.copy()

train_features = pd.get_dummies(train_features)
test_features = pd.get_dummies(test_features)
X, test_final = train_features.align(test_features, join='left', axis=1, fill_value=0)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
#Ridge
ridge_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1.0))
])

ridge_pipe.fit(X_train, y_train)
y_pred = ridge_pipe.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(rmse)

In [ ]:
alphas = [0.01, 0.1, 1, 10, 100]

for a in alphas:
    ridge = Ridge(a)
    ridge.fit(X_train, y_train)
    y_pred = ridge.predict(X_val)
    mse = mean_squared_error(y_val, y_pred)
    rmse = np.sqrt(mse)
    print(rmse)

In [ ]:
#Tried LASSO but no improvement
from sklearn.linear_model import Lasso
lasso_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', Lasso(alpha=0.1, max_iter=10000))
])

lasso_pipe.fit(X_train, y_train)

y_pred = lasso_pipe.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(rmse)
for a in alphas:
    lasso = Lasso(a)
    lasso.fit(X_train, y_train)
    y_pred = lasso.predict(X_val)
    mse = mean_squared_error(y_val, y_pred)
    rmse = np.sqrt(mse)
    print(rmse)
    coef = lasso.coef_
    print('Total:', len(coef))
    print('Selection:', np.sum(coef != 0))

In [ ]:
#Tried non-linear model but no improvement
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)

from sklearn.metrics import mean_squared_error
import numpy as np

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(rmse)

In [ ]:
#Using RidgeCV for optimalization
from sklearn.linear_model import RidgeCV
alphas = [0.75, 0.95, 1, 1.5, 3, 5, 10, 20, 50, 100] 
ridge_cv = RidgeCV(alphas = alphas, cv = 5)
ridge_cv.fit(X, y)
print(ridge_cv.alpha_)
best_alpha = ridge_cv.alpha_

ridge = Ridge(alpha=best_alpha)
ridge.fit(X_train, y_train)

y_pred = ridge.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(rmse)

In [ ]:
#Compare the result from RidgeCV to Ridge
# alpha = 0.1
ridge1 = Ridge(alpha=0.1)
ridge1.fit(X_train, y_train)
y_pred1 = ridge1.predict(X_val)
rmse1 = np.sqrt(mean_squared_error(y_val, y_pred1))

# alpha = 10
ridge2 = Ridge(alpha=10)
ridge2.fit(X_train, y_train)
y_pred2 = ridge2.predict(X_val)
rmse2 = np.sqrt(mean_squared_error(y_val, y_pred2))

print("alpha=0.1:", rmse1)
print("alpha=10 :", rmse2)

# Final model

In [ ]:
final_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=10))
])

final_model.fit(X, y)
test_pred_log = final_model.predict(test_final)
test_pred = np.expm1(test_pred_log)

submission = pd.DataFrame({
    'Id': test['Id'],
    'SalePrice': test_pred
})
submission.to_csv('submission.csv', index=False)
submission.head()

# Results

The Ridge regression model achieved a best RMSE of about 0.125 on the validation set, showing good prediction performance.

Different values of alpha were tested. A smaller alpha (0.1) performed better, while a larger alpha (10) gave worse results. This means too much regularization reduces accuracy.

Feature selection was also tested. When the number of features was reduced a lot, the RMSE increased to around 0.19. This shows that removing too many features leads to loss of important information.

The final model achieved an RMSE of about 0.132, which is close to the best result. Overall, keeping enough features and using a moderate alpha gives better performance.